# 🚀 Mastering AI Agents: Multi-Agent Orchestration with Google ADK! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

Labs and Goals:
- **Lab2: Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **[HERE]>>> Lab 3: Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Lab 4: Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Demo: Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

Credit: Notebook content adapted from Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, who passionate about helping developers build with AI and cloud technologies.



-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: [Google AI Studio](https://aistudio.google.com/app/apikey)

 -----------------------------------------------------------------------------


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
# --- Lab Cell ID 1----#
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Configure Your API Key
To use Gemini models, you need an API key from [Google AI Studio](https://aistudio.google.com/app/apikey). This section securely collects your key and configures it for the ADK.


In [ ]:
# --- Lab Cell ID 2----#
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [ ]:
# --- Lab Cell ID 3----#
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


In [ ]:
# --- Lab Cell ID 6 ----#

# For Part2: You also have to run This Cell together with Cell 1 and Cell 2

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [ ]:
# --- Lab Cell ID 8 ----#

# --- Tool Definition: A function that calls a live public API ---
import requests

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}



In [ ]:
# --- Lab Cell ID 9 ----#
# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast,]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [ ]:
# --- Lab Cell ID 5 ----
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response



In [ ]:
# --- Lab Cell ID 10 ----#

# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: 'd40383a3-5df3-49d8-8ab4-0fd51ed3ffb1'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-406648d3-fde3-47f4-b49f-dbbeb59d64a7',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xde\x02\x01\x0c9\xd6\xc7\xf2\xe5\xa1\xd7\xa4\xc97T\x01\x1cg\x03\x1dV\xf9^Id (X<J-)\x03\x89\xac\xad+\xc51\xe3i\x00w\xdcP\xcby+\xdf\xc8\xd4\xf05\xe8H\x16\x94\xe0\x02\xdd\x06\x9a1u\xfd\xd7\x911\r\xb7e\xe4W\x11z0\xda\x1e\xf6V\xf5\xe1\x8frl\x0e-\x93.\x0b\x82\xbe\x9d-^N...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=187,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=187


The weather near Lake Tahoe is mostly clear with a low around 47°F, and a west wind of 0 to 5 mph. It sounds like good weather for hiking! Enjoy your hike!

--------------------------------------------------



## Assignment : Adding Environmental Awareness 🍃

Now that your agent can check the weather, let's give it another vital skill: **Air Quality Monitoring**. Many outdoor activities depend not just on the forecast, but on the air quality index.

#### The Objective
Create a new tool that fetches real-time **PM2.5** levels for a given location using the [Open-Meteo Air Quality API](https://air-quality-api.open-meteo.com/en/docs/air-quality-api).

#### Steps to Complete
1.  **Review the API:** Visit the [Open-Meteo Air Quality API documentation](https://air-quality-api.open-meteo.com/en/docs/air-quality-api). Note that this API requires `latitude` and `longitude` parameters rather than city names.
2.  **Define the Tool:** Implement a function `get_pm25_level(lat: float, lon: float)` that queries the API and parses the PM2.5 data.
3.  **Refine the Agent:** Add this new tool to your `weather_aware_planner` agent's `tools` list.
4.  **Chain the Logic:** Update your agent's `instruction` prompt to define its goals as a health-conscious planner. Instruct the agent to autonomously evaluate whether it needs to call `get_live_weather_forecast` **OR** `get_pm25_level` (or both) to provide informed safety advice, specifically warning the user if PM2.5 levels exceed 35 µg/m³.

In [ ]:
# --- Lab Cell: PM2.5 Tool Definition ---
def get_pm25_level(lat: float, lon: float) -> dict:
    """
    Gets the current PM2.5 concentration for a specific latitude and longitude.

    Fetches the current PM2.5 air quality level for a specific geographic coordinate.
    Use this tool whenever the user asks about air safety, pollution levels,
    or the suitability of outdoor environments.

    """
    # Open-Meteo Air Quality API endpoint
    url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&current=pm2_5"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Extract the current PM2.5 value
        pm2_5 = data.get('current', {}).get('pm2_5')
        return {
            "status": "success",
            "pm2_5": pm2_5,
            "unit": data.get('current_units', {}).get('pm2_5')
        }
    except Exception as e:
        return {"status": "error", "message": f"Failed to fetch air quality: {e}"}

# HINT--- Lab Cell: Update the Agent ---
# 1. Update the instruction to focus on autonomous tool selection
new_instruction = """
You are a helpful and health-conscious trip planner assistant.
Your goal is to provide comprehensive advice for outdoor activities.

- Use `get_live_weather_forecast` when the user asks about weather conditions.
- Use `get_pm25_level` when the user asks about air safety or pollution levels.

When a user asks about outdoor activities, autonomously evaluate which information is relevant. If you check air quality and find PM2.5 levels > 35 µg/m³, clearly advise the user to exercise caution due to unhealthy air quality.
"""

# 2. Re-define the agent with the updated instructions
weather_agent.instruction = new_instruction
weather_agent.tools = [get_live_weather_forecast, get_pm25_level]

print(f"✅ Agent '{weather_agent.name}' updated with flexible tool selection!")